# Actividad 1 · Interpretación y uso de modelos

**Tiempo estimado:** 20 minutos  
**Condición:** obligatoria

En esta actividad vas a retomar el modelo de clasificación construido durante la Semana 11 para analizar qué significa una predicción. El objetivo no es entrenar un modelo nuevo ni memorizar código, sino diferenciar entre el resultado que produce el modelo, la interpretación de ese resultado y la decisión que podría tomarse en un contexto real.



## Preparación del entorno y reconstrucción del modelo

Ejecutá las siguientes celdas en orden. SQL se utiliza para seleccionar los datos y Python para preparar, entrenar e interpretar el modelo.


In [ ]:
#@title Preparación automática del entorno (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

for package in ["duckdb", "pandas", "scikit-learn"]:
    module = "sklearn" if package == "scikit-learn" else package
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
DATA_PATH = Path("titanic.csv")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)

titanic_original = pd.read_csv(DATA_PATH)
con = duckdb.connect(":memory:")
con.register("titanic_original", titanic_original)
con.execute("CREATE OR REPLACE TABLE titanic AS SELECT * FROM titanic_original")

print(f"Entorno preparado: {len(titanic_original):,} pasajeros y {titanic_original.shape[1]} variables.")


In [ ]:
df = con.sql("""
SELECT
    survived AS objetivo,
    pclass,
    sex,
    age,
    sibsp,
    fare
FROM titanic
""").df()

y = df["objetivo"]
X = df.drop(columns="objetivo")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Los valores faltantes se completan usando solo los datos de entrenamiento.
mediana_edad = X_train["age"].median()
X_train["age"] = X_train["age"].fillna(mediana_edad)
X_test["age"] = X_test["age"].fillna(mediana_edad)

# Las variables categóricas se preparan con las categorías de entrenamiento.
X_train = pd.get_dummies(X_train, columns=["sex"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["sex"], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
probabilidad = modelo.predict_proba(X_test)[:, 1]

print(f"Casos de entrenamiento: {len(X_train):,}")
print(f"Casos de prueba: {len(X_test):,}")


## Predicción, probabilidad y decisión

Una clasificación binaria produce una categoría, pero el modelo también puede estimar una probabilidad. Un valor cercano a 0 o a 1 indica que el modelo se inclina más por una categoría; un valor cercano a 0,5 señala una predicción menos definida.

Esa probabilidad no es una verdad ni una orden de acción. Debe interpretarse considerando el problema, la calidad de los datos, el costo de equivocarse y la información que el modelo no observa.


In [ ]:
resultados = X_test.copy()
resultados["resultado_real"] = y_test.to_numpy()
resultados["prediccion"] = y_pred
resultados["probabilidad_clase_1"] = probabilidad.round(3)
resultados["distancia_a_0_5"] = (resultados["probabilidad_clase_1"] - 0.5).abs()

con.register("predicciones", resultados.reset_index(drop=True))

casos = con.sql("""
SELECT
    resultado_real,
    prediccion,
    probabilidad_clase_1,
    CASE
        WHEN probabilidad_clase_1 < 0.30 THEN 'Probabilidad baja'
        WHEN probabilidad_clase_1 > 0.70 THEN 'Probabilidad alta'
        ELSE 'Caso incierto'
    END AS lectura_inicial
FROM predicciones
ORDER BY distancia_a_0_5 ASC
LIMIT 12
""").df()

display(casos)


### Observá los casos extremos

Compará ahora algunos casos con probabilidades muy bajas y muy altas.


In [ ]:
extremos = con.sql("""
(SELECT resultado_real, prediccion, probabilidad_clase_1
 FROM predicciones ORDER BY probabilidad_clase_1 ASC LIMIT 4)
UNION ALL
(SELECT resultado_real, prediccion, probabilidad_clase_1
 FROM predicciones ORDER BY probabilidad_clase_1 DESC LIMIT 4)
""").df()

display(extremos)


## Actividad de interpretación

Respondé dentro del notebook:

1. ¿Qué diferencia existe entre la categoría predicha y la probabilidad estimada?
2. Elegí un caso cercano a 0,5. ¿Por qué convendría actuar con mayor cautela?
3. ¿Qué información adicional necesitarías antes de transformar una predicción en una decisión?
4. ¿Por qué una probabilidad alta no demuestra que las variables utilizadas causen el resultado?

## Cierre

Una predicción es un resultado del modelo. Interpretarla exige relacionarla con el contexto y reconocer su incertidumbre. Decidir implica, además, considerar objetivos, costos, riesgos y criterios éticos.


In [ ]:
con.close()
